In [62]:
import os

In [2]:
%pwd

'c:\\Users\\lokar\\Desktop\\End-to-End-Machine-Learning-Project-with-ML-FLOW\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\lokar\\Desktop\\End-to-End-Machine-Learning-Project-with-ML-FLOW'

In [5]:
os.environ["ML_FLOW_TRACKING_URI"]="https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="LokeshLokareddy"

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [7]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories, save_json

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([Path(self.config.artifacts_root)])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN


        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            all_params=params,
            metric_file_name=Path(config.metric_file_name),
            target_column=schema.name,
            mlflow_uri=os.environ.get("ML_FLOW_TRACKING_URI"),
        )

        return model_evaluation_config


In [9]:
import os
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [65]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)
        mlflow.set_experiment("Wine Quality Prediction")
        print("MLflow experiment ID:", experiment.experiment_id)

        with mlflow.start_run(experiment_id=experiment.experiment_id ):

            mlflow.sklearn.log_model(
             sk_model=model,
             name="model",
             registered_model_name="ElasticNet",
            )
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2)= self.eval_metrics(test_y, predicted_qualities)


            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data = scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)


            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model")

            else:
                mlflow.sklearn.log_model(model,"model")

In [64]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-11-03 21:59:57,579: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-03 21:59:57,605: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-03 21:59:57,615: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-11-03 21:59:57,631: INFO: common: created directory at: artifacts]
[2026-11-03 21:59:57,635: INFO: common: created directory at: artifacts/model_evaluation]


2026/11/03 21:59:59 INFO mlflow.tracking.fluent: Experiment with name 'Wine Quality Prediction' does not exist. Creating a new experiment.
Registered model 'ElasticNet' already exists. Creating a new version of this model...
2026/11/03 22:00:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet, version 2
Created version '2' of model 'ElasticNet'.


[2026-11-03 22:00:36,382: INFO: common: json file saved at: artifacts\model_evaluation\metric.json]


2026/11/03 22:00:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run placid-quail-169 at: https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow/#/experiments/1/runs/27e9885566684599b17dfe2406e71f8f
🧪 View experiment at: https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow/#/experiments/1


In [54]:
import os
import mlflow
import mlflow.sklearn
from getpass import getpass

tracking_uri = os.environ["ML_FLOW_TRACKING_URI"]  # Raises KeyError if not set
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

os.environ["MLFLOW_TRACKING_USERNAME"] = "LokeshLokareddy"
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("DagsHub token: ")


In [48]:
import os
import mlflow
from getpass import getpass

os.environ["MLFLOW_TRACKING_USERNAME"] = "LokeshLokareddy"
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("DagsHub access token: ")

tracking_uri = "https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow"
os.environ["ML_FLOW_TRACKING_URI"] = tracking_uri

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)


print("Tracking URI:", mlflow.get_tracking_uri())
print("Username set:", bool(os.environ.get("MLFLOW_TRACKING_USERNAME")))
print("Token set:", bool(os.environ.get("MLFLOW_TRACKING_PASSWORD")))

Tracking URI: https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow
Username set: True
Token set: True


In [51]:
import os
import mlflow
from getpass import getpass

os.environ["MLFLOW_TRACKING_USERNAME"] = "LokeshLokareddy"
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("DagsHub access token: ")

tracking_uri = os.environ["ML_FLOW_TRACKING_URI"]
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

In [59]:
%pip install dagshub
import dagshub
dagshub.init(repo_owner='LokeshLokareddy', repo_name='End-to-End-Machine-Learning-Project-with-ML-FLOW', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

  Using cached dagshub-0.7.3-py3-none-any.whl.metadata (12 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached gql-4.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.3.1-py3-none-any.whl.metadata (12 kB)
  Using cached semver-3.1.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached dagshub_annotation_converter-0.3.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached lxml-6.1.3-cp312-cp312-win_amd64.whl.metadata (3.4 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphene 3.4.3 requires graphql-core<3.3,>=3.1, but you have graphql-core 3.3.1 which is incompatible.
graphql-relay 3.2.0 requires graphql-core<3.3,>=3.2, but you have graphql-core 3.3.1 which is incompatible.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=cbf1b995-58db-4052-87b8-5a958a57a334&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f9255df0389b858cc7b3c778a0b6aba3ecab6271a1340cc5cbaf9d79998d7eb3




[2026-11-03 19:27:54,280: INFO: _client: HTTP Request: POST https://dagshub.com/login/oauth/middleman "HTTP/1.1 200 OK"]


... (3 outputs left out)

Initialized MLflow to track repo "LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW"

[2026-11-03 19:28:03,316: INFO: helpers: Initialized MLflow to track repo "LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW"]


Repository LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW initialized!

[2026-11-03 19:28:03,319: INFO: helpers: Repository LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW initialized!]
🏃 View run indecisive-snipe-957 at: https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow/#/experiments/0/runs/b2c0bac8515146809d0c789c3f37c77e
🧪 View experiment at: https://dagshub.com/LokeshLokareddy/End-to-End-Machine-Learning-Project-with-ML-FLOW.mlflow/#/experiments/0
